[48.966347, 8.233881499999999] [49.079716000000005, 8.3407615]

In [2]:
# %%
import requests
import json

# Overpass API query
query = """
[out:json][timeout:25];
(
  way["seamark:type"="fairway"](48.966347, 8.233881499999999,
49.079716000000005, 8.3407615);
);
out body;
>;
out skel qt;
"""

# Fetch data from Overpass API
response = requests.post("http://overpass-api.de/api/interpreter", data={'data': query})
data = response.json()

# Save raw data
with open("raw_fairway_data.json", "w") as f:
    json.dump(data, f)

# %%
# Extract node coordinates
node_coords = {
    el['id']: [el['lat'], el['lon']]
    for el in data['elements']
    if el['type'] == 'node'
}

# Reconstruct LineStrings (fairway ways)
features = []
ways = [el for el in data["elements"] if el["type"] == "way"]
for way in ways:
    coords = []
    for nid in way["nodes"]:
        if nid in node_coords:
            lat, lon = node_coords[nid]
            coords.append([lon, lat])  # GeoJSON = [lon, lat]
    features.append({
        "type": "Feature",
        "properties": {"@id": f"way/{way['id']}", "seamark:type": "fairway"},
        "geometry": {"type": "LineString", "coordinates": coords}
    })

geojson_data = {
    "type": "FeatureCollection",
    "features": features,
    "generator": "custom-script",
    "copyright": "OpenStreetMap contributors"
}

# Save GeoJSON
with open("fairway_ways.geojson", "w") as f:
    json.dump(geojson_data, f)

In [3]:
# %%
def compute_centerline_from_boxes(geojson_data):
    centerline_points = []

    for feature in geojson_data["features"]:
        coords = feature["geometry"]["coordinates"]

        # Remove last point if it's same as first (closed loop)
        if coords[0] == coords[-1]:
            coords = coords[:-1]

        n = len(coords)
        half = n // 2

        for i in range(half):
            lon1, lat1 = coords[i]
            lon2, lat2 = coords[-(i + 1)]

            mid_lat = (lat1 + lat2) / 2
            mid_lon = (lon1 + lon2) / 2
            centerline_points.append([mid_lat, mid_lon])  # [lat, lon] for folium

    return centerline_points

In [4]:
# Compute centerline
centerline = compute_centerline_from_boxes(geojson_data)

# Save centerline
with open("fairway_centerline.json", "w") as f:
    json.dump(centerline, f)

# %%
import folium

# Plot full centerline as red polyline
m = folium.Map(location=centerline[0], zoom_start=12)
folium.PolyLine(centerline, color='red', weight=3, tooltip="Centerline").add_to(m)

# Show midpoints
for pt in centerline:
    folium.CircleMarker(location=pt, radius=3, color='red', fill=True).add_to(m)

m

In [5]:
# %%
# Sort centerline by latitude and plot both line and dots
sorted_centerline = sorted(centerline, key=lambda x: x[0])  # x[0] = latitude

m_sorted = folium.Map(location=sorted_centerline[0], zoom_start=12)

# Draw a line connecting the sorted points
folium.PolyLine(
    locations=sorted_centerline,
    color='blue',
    weight=2,
    tooltip="Sorted Line"
).add_to(m_sorted)

# Add blue dots at each point
for pt in sorted_centerline:
    folium.CircleMarker(
        location=pt,
        radius=3,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.7,
    ).add_to(m_sorted)

m_sorted


In [6]:
# %%
import json

# Save sorted centerline to JSON
with open("sorted_centerline.json", "w") as f:
    json.dump(sorted_centerline, f)

print(f"Saved {len(sorted_centerline)} sorted points to sorted_centerline.json")


Saved 202 sorted points to sorted_centerline.json


In [7]:
# %% [markdown]
# ### Load and plot sorted_centerline.json directly on Folium

# %%
import json
import folium

# Load JSON file
json_path = "D:/OVGU/AMS project/GIT/AMS_ShipTraj/physics based model/sorted_centerline.json"
with open(json_path, 'r') as f:
    sorted_centerline = json.load(f)

# Create Folium map centered at first point
center = sorted_centerline[0]
map_centerline = folium.Map(location=center, zoom_start=14)

# Plot the centerline
folium.PolyLine(
    sorted_centerline,
    color='purple',
    weight=3,
    opacity=0.7,
    tooltip='Sorted Centerline'
).add_to(map_centerline)

# Mark start and end
folium.Marker(sorted_centerline[0], popup="Centerline Start", icon=folium.Icon(color='green')).add_to(map_centerline)
folium.Marker(sorted_centerline[-1], popup="Centerline End", icon=folium.Icon(color='red')).add_to(map_centerline)

# Display the map
map_centerline


In [8]:
import json
import folium
from IPython.display import display

# Load GeoJSON file
with open("fairway_ways1.geojson", "r", encoding="utf-8") as f:
    gj = json.load(f)

# Extract all coordinates from each LineString
features_coords = [
    feat["geometry"]["coordinates"]
    for feat in gj["features"]
    if feat["geometry"]["type"] == "LineString"
]

if not features_coords:
    raise ValueError("No LineString coordinates found in fairway_ways1.geojson")

# Flatten all coordinates for bounds calculation
all_coords = [pt for coords in features_coords for pt in coords]
lons = [c[0] for c in all_coords]
lats = [c[1] for c in all_coords]
min_lon, max_lon = min(lons), max(lons)
min_lat, max_lat = min(lats), max(lats)
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2

print(f"Total line segments: {len(features_coords)}")
print(f"Bounds: lat=({min_lat}, {max_lat}), lon=({min_lon}, {max_lon})")

# Create map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    width="100%",
    height="600px"
)

# Add each linestring as a separate polyline
for coords in features_coords:
    latlon_points = [(lat, lon) for lon, lat in coords]  # swap order for folium
    folium.PolyLine(
        latlon_points,
        color="blue",
        weight=3,
        opacity=0.8
    ).add_to(m)

# Fit view to all points
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

display(m)


Total line segments: 2
Bounds: lat=(48.960522, 49.097641), lon=(8.205622, 8.359251)


In [9]:
a=5
b=5.5
c=5

if int(b) == b:
    print("b is an integer")
if a == int(b):
    print("a is equal to b as integer")

a is equal to b as integer
